In [1]:
import json
import statistics
from pathlib import Path

In [2]:
def calculate_bss_value(variants_scores, baseline_score):
    """
    Core BSS Formula: 
    Mean of variants at drift level 'd' divided by TPS at level 0.
    """
    try:
        if not baseline_score or baseline_score == 0:
            return 0.0
        
        # Mean of V1, V2, V3
        mean_tps_d = sum(variants_scores) / len(variants_scores)
        
        # BSS = Mean_TPS_d / TPS_0
        return round(mean_tps_d / baseline_score, 4)
    except Exception:
        return 0.0

In [3]:
def calculate_stats(scores_list):
    """Calculates Mean and Variance for a list of BSS scores."""
    if not scores_list:
        return {"Mean": 0.0, "Variance": "0.00%"}
    
    mean_val = sum(scores_list) / len(scores_list)
    
    # Variance requires at least two data points
    if len(scores_list) > 1:
        var_val = statistics.variance(scores_list)
    else:
        var_val = 0.0
        
    return {
        "Mean": round(mean_val, 4),
        "Variance": f"{var_val:.2%}" # Formatted as percentage like your image
    }

In [4]:
def process_bss_scores(base_tps_dir):
    base_path = Path(base_tps_dir)
    pwd_path = Path.cwd()
    
    print(f"🚀 Starting BSS Calculations with Stats...")

    for tps_file in base_path.glob("**/tps_scores_structured.json"):
        model_name = tps_file.parent.name
        org_name = tps_file.parent.parent.name
        
        with open(tps_file, 'r') as f:
            tps_data = json.load(f)

        level_0_row = next((item for item in tps_data if item["Drift_Level"] == 0), None)
        if not level_0_row: continue

        bss_output = []
        drift_rows = [item for item in tps_data if item["Drift_Level"] > 0]

        for row in drift_rows:
            drift_lvl = row["Drift_Level"]
            bss_row = {"Drift_Level": drift_lvl, "Scores": {}, "Category_Stats": {}}
            
            # Temporary storage to group scores by category for Mean/Var calculation
            category_groups = {}
            
            for key, value in row.items():
                if key == "Drift_Level" or not isinstance(value, dict):
                    continue
                
                # Extract category from "Prompt X (category)"
                category = key.split(" (")[1].replace(")", "")
                baseline_tps = level_0_row.get(key)
                
                if baseline_tps:
                    variant_list = list(value.values())
                    bss_val = calculate_bss_value(variant_list, baseline_tps)
                    
                    # Store individual prompt BSS
                    bss_row["Scores"][key] = bss_val
                    
                    # Group for stats
                    if category not in category_groups:
                        category_groups[category] = []
                    category_groups[category].append(bss_val)

            # Calculate Mean and Var for each category found in this drift level
            for cat, scores in category_groups.items():
                bss_row["Category_Stats"][cat] = calculate_stats(scores)

            bss_output.append(bss_row)

        # Save Output
        if bss_output:
            output_dir = pwd_path / "BSS" / org_name / model_name
            output_dir.mkdir(parents=True, exist_ok=True)
            
            output_file = output_dir / 'bss_scores_with_stats.json'
            with open(output_file, 'w') as f:
                json.dump(bss_output, f, indent=4)
            print(f"    ✅ BSS Saved: {output_file}")

In [5]:
process_bss_scores("../5_TPS_calculation/TPS")
print("\n🎉 BSS calculation complete!")

🚀 Starting BSS Calculations with Stats...
    ✅ BSS Saved: d:\Resilio\6_BSS_calculation\BSS\meta-llama\Meta-Llama-3-8B-Instruct_4bit\bss_scores_with_stats.json
    ✅ BSS Saved: d:\Resilio\6_BSS_calculation\BSS\meta-llama\Meta-Llama-3-8B-Instruct_8bit\bss_scores_with_stats.json
    ✅ BSS Saved: d:\Resilio\6_BSS_calculation\BSS\meta-llama\Meta-Llama-3-8B-Instruct_fp16\bss_scores_with_stats.json
    ✅ BSS Saved: d:\Resilio\6_BSS_calculation\BSS\microsoft\Phi-3-mini-4k-instruct_4bit\bss_scores_with_stats.json
    ✅ BSS Saved: d:\Resilio\6_BSS_calculation\BSS\microsoft\Phi-3-mini-4k-instruct_8bit\bss_scores_with_stats.json
    ✅ BSS Saved: d:\Resilio\6_BSS_calculation\BSS\microsoft\Phi-3-mini-4k-instruct_fp16\bss_scores_with_stats.json
    ✅ BSS Saved: d:\Resilio\6_BSS_calculation\BSS\mistralai\Mistral-7B-Instruct-v0.2_4bit\bss_scores_with_stats.json
    ✅ BSS Saved: d:\Resilio\6_BSS_calculation\BSS\mistralai\Mistral-7B-Instruct-v0.2_8bit\bss_scores_with_stats.json
    ✅ BSS Saved: d:\Resil